In [ ]:
#from torch import nn
import pandas as pd
import numpy as np
from pathlib import Path
import torch
from nfl.lib import enums

from nfl.data_management.DataManager import DataManager
from nfl.NeuralNetwork.EPA_Predictor import EPAPredictor
from nfl.NeuralNetwork.NNSolver import Solver
from nfl.lib.utils import create_train_test_split
from torch.utils.tensorboard import SummaryWriter
from nfl.NeuralNetwork.parameters.ParameterSelector import GridSearch, GeneticSearch
from nfl.NeuralNetwork.parameters.OptunaSearch import OptunaSearch

In [ ]:
SCENARIO_COLS = [
    "yardline_100",
    "game_seconds_remaining",
    "has_turf",
    "temp",
    "wind",
    "has_roof",
    "ydstogo",
    "goal_to_go",
    "score_differential",
    "down",
    "div_game",
    "day_of_season",
    "series"
]

PLAY_COLS = [
    "play_type",
    "pass_location",
#    "pass_length",
#    "run_location",
    "run_gap",
    "shotgun",
    "no_huddle",
    "qb_kneel",
    "qb_spike",
    "qb_scramble",
    "air_yards"
]

RESULT_COLS = [
    "epa",
    "wpa",
    "success",
    "result",
    "series_success",
    "tackle_for_loss",
    "saftey",
    "yards_gained",
    "touchdown",
    "fumble",
    "complete_pass",
    "rushing_yards",
    "fumble_lost",
    "interception",
    "sack",
    "penalty_yards",
]

EXCLUDED_PLAY_TYPES = {
    enums.PlayType.KICK,
    enums.PlayType.EXTRA_POINT,
    enums.PlayType.NO_PLAY,
    enums.PlayType.GAME_START,
}

In [ ]:
data = DataManager.get_data(path_to_json = (Path.cwd() / "nfl/data").resolve())
data = DataManager.clean_data(data, excluded_play_types=EXCLUDED_PLAY_TYPES, target_col="epa")
data, feature_cols = DataManager.prepare_features(data, scenario_columns=SCENARIO_COLS, play_columns=PLAY_COLS)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Cuda device count: ", torch.cuda.device_count())
print(f"Using {device} device")

In [ ]:
pd.set_option('display.max_columns', None)
data.head(5)

In [ ]:
split = create_train_test_split(df=data, feature_cols=feature_cols, target_col="epa", train_frac = 0.8, device=device)
X_train, y_train, X_test, y_test, scaler = split

In [ ]:
EPA_INPUT_SIZE = len(feature_cols)
NUM_ITER = 7
NUM_EPOCH = 50
EPS = 1e-8
BETAS = (0.9, 0.999)

MIN_HIDDEN_LAYERS, MAX_HIDDEN_LAYERS = 1, 20
MIN_HIDDEN_SIZE, MAX_HIDDEN_SIZE = 16, 256
MIN_LEARNING_RATE, MAX_LEARNING_RATE = 1e-5, 1e-2
MIN_BATCH_SIZE, MAX_BATCH_SIZE = 64, 512
MIN_WEIGHT_DECAY, MAX_WEIGHT_DECAY = 1e-5, 1e-2

BATCH_SIZE = np.linspace(MIN_BATCH_SIZE, MAX_BATCH_SIZE, num=NUM_ITER).astype(int).tolist()
LEARNING_RATE = np.geomspace(MIN_LEARNING_RATE, MAX_LEARNING_RATE, num=NUM_ITER).tolist()
NUM_HIDDEN_LAYERS = np.linspace(MIN_HIDDEN_LAYERS, MAX_HIDDEN_LAYERS, num=NUM_ITER).astype(int).tolist()
HIDDEN_SIZE = np.linspace(MIN_HIDDEN_SIZE, MAX_HIDDEN_SIZE, num=NUM_ITER).astype(int).tolist()
WEIGHT_DECAY = np.geomspace(MIN_WEIGHT_DECAY, MAX_WEIGHT_DECAY, num=NUM_ITER).tolist()

param_grid = {
    "batch_size": BATCH_SIZE,
    "lr": LEARNING_RATE,
    "num_hidden_layers": NUM_HIDDEN_LAYERS,
    "hidden_size": HIDDEN_SIZE,
    "weight_decay": WEIGHT_DECAY,
}

In [ ]:
pop_size = len(param_grid)*3
top_k = pop_size//4
num_generations = 4*len(param_grid)

ga_search = GeneticSearch(
    param_grid=param_grid,
    solver_cls=Solver,
    model_cls=EPAPredictor,
    criterion=torch.nn.MSELoss().to(device),
    device=device,
    num_epochs=NUM_EPOCH,
    input_size=EPA_INPUT_SIZE,
    pop_size=pop_size,
    generations=num_generations,
    mutation_rate=0.25,
    top_k=top_k,
    log_dir="runs/ga_sweep"
)

results_df, best_config = ga_search.run(X_train, y_train, X_test, y_test, metric="best_val_loss")

print("\nBest Model Found:")
print(best_config)

# 2. Save results to CSV for inspection
results_df.to_csv("ga_sweep_results.csv", index=False)

In [ ]:
optuna_search = OptunaSearch(
    param_grid=param_grid,
    solver_cls=Solver,
    model_cls=EPAPredictor,
    criterion=torch.nn.MSELoss().to(device),
    device=device,
    num_epochs=NUM_EPOCH,
    input_size=EPA_INPUT_SIZE,
    log_dir="runs/optuna_sweep",
    prune=True
)
# Runs 50 trials (Bayesian sampling + pruning poor runs early)
results_df, best_config = optuna_search.run(X_train, y_train, X_test, y_test, n_trials=50)

print("\nBest Hyperparameters Found:")
print(best_config)